In [ ]:
from utils_sm import *
from utils_SDEMEM_realdata import *
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import numpy as np
import torch
import math
import random
import matplotlib.pyplot as plt
import torch.optim as optim
from tqdm import tqdm
import time

In [ ]:
obs_size = 40

In [ ]:
# ========== load observed data
df = pd.read_excel("realdata/20160427_mean_eGFP.xlsx", header=None)
x_obs = torch.tensor(df.to_numpy(), dtype=torch.float32)[:, 1:].T.log() 

x_obs = x_obs[:obs_size]

In [ ]:
# ===== Setting for real data ===== #
var_name = ['log_m0', 'log_scale', 'log_offset', 'log_sigma', 'mu_delta', 'mu_gamma', 'mu_k', 'mu_t0', 'log_tau_delta', 'log_tau_gamma', 'log_tau_k', 'log_tau_t0']

# The first 8 parameters have normal priors
prior_mean = torch.tensor([5, 1, 3, -1, -1, -5, 0.5, 0], dtype = torch.float32).unsqueeze(0).to(device)
prior_std = torch.tensor([1, 1, 1, 1, 1, 2, 1, 1], dtype = torch.float32).unsqueeze(0).to(device)

# For the last 4 parameters, the precision follows Gamma priors
prior_alpha = torch.tensor([2, 2, 2, 2], dtype = torch.float32).unsqueeze(0).to(device)
prior_beta = torch.tensor([0.5, 0.5, 0.5, 0.5], dtype = torch.float32).unsqueeze(0).to(device)

T = 30
theta_dim = 12
x_dim = 180 

## Density plot

In [ ]:
def normalized_weight(log_w):
    tmp_w = (log_w - log_w.max()).exp()
    tmp_w = tmp_w / tmp_w.sum()

    thresh_id = 0
    while tmp_w.max() > 100 / log_w.shape[0]:
        thresh_id += 1
        clip = log_w.sort(descending=True).values[thresh_id]
        log_w = torch.clamp(log_w, max = clip)
        tmp_w = (log_w - log_w.max()).exp()
        tmp_w = tmp_w / tmp_w.sum()
    return tmp_w

In [ ]:
# ===== Load results ===== #
theta_ours = torch.from_numpy(np.load(f"sample_res_all/theta_post_precond_mchain_sameprior_task{0}.npy"))
theta_NPE = torch.from_numpy(np.load(f"sample_res_all/theta_post_NPE_newdeepsets_task{0}.npy")) 

In [ ]:
theta_set_ABC_all = []
W1_set_all = []
for i in range(10):
    theta_set_ABC_all.append(np.load(f'res_ABC/theta_set_batch{i}.npy'))
    W1_set_all.append(np.load(f'res_ABC/W1_set_batch{i}.npy'))

theta_set_ABC = torch.from_numpy(np.concatenate(theta_set_ABC_all))
W1_set_ABC = torch.from_numpy(np.concatenate(W1_set_all))

# W1_set_ABC contains some exact zero. This is because ot.emd2 returns 0 when there is nan in the data. So we exclude those.
nonzero_idx = torch.where(W1_set_ABC != 0)[0]
W1_set_ABC = W1_set_ABC[nonzero_idx]
theta_set_ABC = theta_set_ABC[nonzero_idx]

smallest_values, smallest_indices = torch.topk(W1_set_ABC, 1000, largest=False)  
theta_ABC = theta_set_ABC[smallest_indices].clone()

In [ ]:
theta_ours.shape, theta_NPE.shape, theta_ABC.shape

In [ ]:
theta_ours_aug = torch.cat([theta_ours, (theta_ours[:, 0] + theta_ours[:, 6]).unsqueeze(1), (theta_ours[:, 0] + theta_ours[:, 1] + theta_ours[:, 6]).unsqueeze(1)], dim = 1)
theta_NPE_aug = torch.cat([theta_NPE, (theta_NPE[:, 0]+ theta_NPE[:, 6]).unsqueeze(1), (theta_NPE[:, 0] + theta_NPE[:, 1] + theta_NPE[:, 6]).unsqueeze(1)], dim=1)
theta_ABC_aug = torch.cat([theta_ABC, (theta_ABC[:, 0]+ theta_ABC[:, 6]).unsqueeze(1), (theta_ABC[:, 0] + theta_ABC[:, 1] + theta_ABC[:, 6]).unsqueeze(1)], dim=1)

In [ ]:
# Load SW data
theta_SW1 = np.load("res_SW1/theta_SW1.npy")
loss_SW1 = np.load("res_SW1/final_loss.npy")

nan_idx = np.isnan(theta_SW1).any(axis=1)
theta_SW1 = theta_SW1[~nan_idx]
loss_SW1 = loss_SW1[~nan_idx]

theta_SW1 = torch.tensor(theta_SW1, dtype=torch.float32)[:100]
print(f"theta_SW1.shape = {theta_SW1.shape}")


prop_mean = theta_SW1.mean(dim = 0, keepdims = True)
prop_std = theta_SW1.std(dim = 0, keepdims = True) 

# inflate the proposal std
prop_std *= 2

# the previous prop_std is too small for these two dimensions
prop_std[0, 0] *= 3
prop_std[0, 6] *= 3


prop_std = prop_std.clamp_min(1e-8)
print(f"Using prop_std = {prop_std}")

In [ ]:
from torch.distributions import MultivariateNormal
from torch.distributions import Gamma

# the proposal distribution
dist_prop = MultivariateNormal(loc = prop_mean.ravel().cpu(), covariance_matrix = torch.diag(prop_std.cpu().ravel()**2))
dist_prior_first8 = MultivariateNormal(loc = prior_mean.ravel().cpu(), covariance_matrix = torch.diag(prior_std.cpu().ravel()**2))
dist_prior_last4 = Gamma(concentration = prior_alpha.ravel().cpu().detach(), rate = prior_beta.ravel().cpu().detach())

### Weight for NPE
log_weight_NPE = dist_prior_first8.log_prob(theta_NPE[:, :8]) + dist_prior_last4.log_prob(torch.exp(-2 * theta_NPE[:, 8:12])).sum(dim = 1) + 4 * math.log(2) -2 * theta_NPE[:, 8:12].sum(dim = 1) - dist_prop.log_prob(theta_NPE)
weight_NPE = normalized_weight(log_weight_NPE)

### Weight for ABC
log_weight_ABC = dist_prior_first8.log_prob(theta_ABC[:, :8]) + dist_prior_last4.log_prob(torch.exp(-2 * theta_ABC[:, 8:12])).sum(dim = 1) + 4 * math.log(2) -2 * theta_ABC[:, 8:12].sum(dim = 1) - dist_prop.log_prob(theta_ABC)
weight_ABC = normalized_weight(log_weight_ABC)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Convert to numpy
theta_ours_np = theta_ours_aug.detach().cpu().numpy()
theta_NPE_np  = theta_NPE_aug.detach().cpu().numpy()
theta_ABC_np = theta_ABC_aug.detach().cpu().numpy()

methods = ["single-model", "NPE", "ABC"]
colors = ["tab:blue", "tab:red", "tab:green"]
data_list = [theta_ours_np, theta_NPE_np, theta_ABC_np]
weight_list = [np.ones(theta_ours_np.shape[0]) / theta_ours_np.shape[0],
                weight_NPE.cpu().numpy(),
                weight_ABC.cpu().numpy()]

bw_list = [1.0, 1.0, 1.5]

var_name_aug_latex = [
    r"$\log m_0$",
    r"$\log \mathrm{scale}$",
    r"$\log \mathrm{offset}$",
    r"$\log \sigma$",
    r"$\mu_{\delta}$",
    r"$\mu_{\gamma}$",
    r"$\mu_{k}$",
    r"$\mu_{t_0}$",
    r"$\log \tau_{\delta}$",
    r"$\log \tau_{\gamma}$",
    r"$\log \tau_{k}$",
    r"$\log \tau_{t_0}$",
    r"$\log m_0 + \mu_k$",
    r"$\log \mathrm{scale} + \log m_0 + \mu_k$",
]

plot_dims = [d for d in range(theta_dim + 2) if d not in [0, 1, 6, 12]]


xlim_list = [
    None,          # d = 0, not plotted
    (-4, 5),       # d = 1
    (1.75, 2.25),       # d = 2
    (-4.5, -1.35),       # d = 3
    (-5, 0),        # d = 4
    (-9.99, -1),        # d = 5
    None,          # d = 6, not plotted
    (-0.7, 0.55),       # d = 7
    (-3, 1),       # d = 8
    (-3, 1.5),       # d = 9
    (-2, 0.41),       # d = 10
    (-1.5, -0.1),       # d = 11
    (0, 10),       # d = 12
    (4.5, 7.3),       # d = 13
]


q_low, q_high = 0.001, 0.999
x_margin_ratio = 0.1

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
axes = axes.ravel()

for plot_i, d in enumerate(plot_dims):
    ax = axes[plot_i]

    pooled_d = np.concatenate([data[:, d] for data in data_list])
    pooled_d = pooled_d[np.isfinite(pooled_d)]

    x_low, x_high = np.quantile(pooled_d, [q_low, q_high])

    x_margin = x_margin_ratio * (x_high - x_low)
    x_low -= x_margin
    x_high += x_margin

    for j, data in enumerate(data_list):
        sns.kdeplot(
            x = data[:, d],
            weights=weight_list[j],
            ax=ax,
            fill=True,
            alpha=0.10,
            bw_adjust=bw_list[j],
            label=methods[j],
            color=colors[j],
            linewidth=2,
            cut=0
        )


    # ax.set_xlim(x_low, x_high)
    if xlim_list[d] is None:
        ax.set_xlim(x_low, x_high)   # automatically calculated range
    else:
        ax.set_xlim(xlim_list[d])    # manually specified range

    ax.set_title(var_name_aug_latex[d], fontsize=24)
    ax.set_ylabel("")              # remove Density label
    ax.set_yticklabels([])         # remove y-axis numbers, keep ticks

    ax.tick_params(axis="x", labelsize=16)
    ax.tick_params(axis="y", length=4)

for k in range(len(plot_dims), len(axes)):
    axes[k].axis("off")


handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles, labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=3,
    fontsize=22,
    frameon=False
)

plt.subplots_adjust(
    left=0.035,
    right=0.995,
    bottom=0.22,
    top=0.95,
    wspace=0.1,
    hspace=0.5
)

plt.savefig("Figs/SDEMEM_realdata_kde.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
theta_NLE = torch.from_numpy(np.load(f"res_NLE/theta_post_sameprior.npy"))
theta_NLE_aug = torch.cat([theta_NLE, (theta_NLE[:, 0]+ theta_NLE[:, 6]).unsqueeze(1), (theta_NLE[:, 0] + theta_NLE[:, 1] + theta_NLE[:, 6]).unsqueeze(1)], dim=1)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Convert to numpy
theta_NLE_np = theta_NLE_aug.detach().cpu().numpy()

methods = ["NLE"]
colors = ["tab:blue"]
data_list = [theta_NLE_np]
weight_list = [np.ones(theta_NLE_np.shape[0]) / theta_NLE_np.shape[0]]

bw_list = [1.0]

var_name_aug_latex = [
    r"$\log m_0$",
    r"$\log \mathrm{scale}$",
    r"$\log \mathrm{offset}$",
    r"$\log \sigma$",
    r"$\mu_{\delta}$",
    r"$\mu_{\gamma}$",
    r"$\mu_{k}$",
    r"$\mu_{t_0}$",
    r"$\log \tau_{\delta}$",
    r"$\log \tau_{\gamma}$",
    r"$\log \tau_{k}$",
    r"$\log \tau_{t_0}$",
    r"$\log m_0 + \mu_k$",
    r"$\log \mathrm{scale} + \log m_0 + \mu_k$",
]

plot_dims = [d for d in range(theta_dim + 2) if d not in [0, 1, 6, 12]]



q_low, q_high = 0.001, 0.999
x_margin_ratio = 0.1

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
axes = axes.ravel()

for plot_i, d in enumerate(plot_dims):
    ax = axes[plot_i]

    pooled_d = np.concatenate([data[:, d] for data in data_list])
    pooled_d = pooled_d[np.isfinite(pooled_d)]

    x_low, x_high = np.quantile(pooled_d, [q_low, q_high])

    x_margin = x_margin_ratio * (x_high - x_low)
    x_low -= x_margin
    x_high += x_margin

    for j, data in enumerate(data_list):
        sns.kdeplot(
            x = data[:, d],
            weights=weight_list[j],
            ax=ax,
            fill=True,
            alpha=0.10,
            bw_adjust=bw_list[j],
            label=methods[j],
            color=colors[j],
            linewidth=2,
            cut=0
        )


    ax.set_title(var_name_aug_latex[d], fontsize=24)
    ax.set_ylabel("")              # remove Density label
    ax.set_yticklabels([])         # remove y-axis numbers, keep ticks

    ax.tick_params(axis="x", labelsize=16)
    ax.tick_params(axis="y", length=4)

for k in range(len(plot_dims), len(axes)):
    axes[k].axis("off")


handles, labels = axes[0].get_legend_handles_labels()


plt.subplots_adjust(
    left=0.035,
    right=0.995,
    bottom=0.22,
    top=0.95,
    wspace=0.1,
    hspace=0.5
)

plt.savefig("Figs/NLE_res_SDEMEM_realdata_kde.pdf", dpi=300, bbox_inches="tight")
plt.show()